**Step 1: Install**

In this step, we install the necessary Python libraries required to develop and run the FitBuddy application. These libraries support backend development, web server execution, HTML templates, and form handling.

In [21]:
!pip install -q fastapi uvicorn jinja2 python-multipart markdown google-genai transformers torch

In [22]:
import app as fit_app

text, engine = fit_app.ask_ai("Give a 3 line beginner workout plan")
print("Engine:", engine)
print(text)

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Engine: Qwen local model (no key)
Sure! Here's a 3-day beginner workout plan that you can follow:

Day 1:
- Warm-up: 5 minutes of light cardio (e.g., walking, jogging in place)
- Strength training:
  - Bicep curls: 4 sets of 8-12 reps
  - Tricep dips: 4 sets of 10-15 reps
  - Shoulders press: 4 sets of 12-15 reps
- Cardio:
  - Jumping jacks for 30 seconds
  - Mountain climbers for 30 seconds

Day 2:
- Rest day or low-intensity exercise session
- Cool-down and stretching: 5 minutes of gentle stretching

Day 3:
- Warm-up: 5 minutes of light cardio (e.g., walking, jogging in place)
- Strength training:
  - Chest press: 4 sets of 8-12 reps
  - Back extensions: 4 sets of 10-15 reps
  - Leg presses: 4 sets of 12-15 reps
- Cardio:
  - Cycling or elliptical machine at 30-60 minutes
  - Walking on flat surfaces for 30 minutes

This plan is designed to be challenging but manageable for beginners. Remember to listen to your body and adjust the intensity based on how you feel during the workouts. 

**Step 2: Templates folder + index.html**

In this step, we create a templates folder to store the HTML files of our FitBuddy application. The index.html file is used to design the user interface where users can enter their fitness details and view their generated fitness plans.

In [23]:
import os
os.makedirs("templates", exist_ok=True)

html = """<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Fit Buddy</title>
  <style>
    body { font-family: Arial, sans-serif; background: #eef7f1; margin: 0; padding: 24px 12px; }
    .card { max-width: 720px; margin: auto; background: #fff; padding: 26px; border-radius: 14px; box-shadow: 0 2px 10px rgba(0,0,0,.1); }
    h1 { margin-top: 0; color: #166534; }
    .grid { display: grid; grid-template-columns: 1fr 1fr; gap: 12px; }
    label { font-size: 13px; color: #444; display: block; margin-bottom: 4px; }
    input, select { width: 100%; padding: 9px; border: 1px solid #ccc; border-radius: 8px; box-sizing: border-box; font-size: 14px; }
    button { margin-top: 16px; width: 100%; padding: 12px; background: #16a34a; color: #fff; border: 0; border-radius: 8px; font-size: 16px; cursor: pointer; }
    button:hover { background: #15803d; }
    .result { margin-top: 24px; padding: 16px; background: #f6fbf8; border-left: 4px solid #16a34a; border-radius: 6px; overflow-x: auto; }
    .engine { font-size: 12px; color: #666; margin-bottom: 8px; }
    table { border-collapse: collapse; } td, th { border: 1px solid #ccc; padding: 6px 10px; }
  </style>
</head>
<body>
  <div class="card">
    <h1>💪 Fit Buddy</h1>
    <p>Give your details,and your personal fitness plan will be ready!</p>
    <form method="post" action="generate">
      <div class="grid">
        <div><label>Age</label><input type="number" name="age" min="10" max="90" value="{{ f.age or '' }}" required></div>
        <div><label>Gender</label>
          <select name="gender">
            {% for g in ['Female','Male','Other'] %}<option {% if f.gender == g %}selected{% endif %}>{{ g }}</option>{% endfor %}
          </select></div>
        <div><label>Weight (kg)</label><input type="number" name="weight" value="{{ f.weight or '' }}" required></div>
        <div><label>Height (cm)</label><input type="number" name="height" value="{{ f.height or '' }}" required></div>
        <div><label>Goal</label>
          <select name="goal">
            {% for g in ['Weight Loss','Muscle Gain','Stay Fit','Improve Stamina'] %}<option {% if f.goal == g %}selected{% endif %}>{{ g }}</option>{% endfor %}
          </select></div>
        <div><label>Fitness Level</label>
          <select name="level">
            {% for g in ['Beginner','Intermediate','Advanced'] %}<option {% if f.level == g %}selected{% endif %}>{{ g }}</option>{% endfor %}
          </select></div>
        <div><label>Workout days per week</label>
          <select name="days">
            {% for g in ['3','4','5','6'] %}<option {% if f.days == g %}selected{% endif %}>{{ g }}</option>{% endfor %}
          </select></div>
        <div><label>Equipment</label>
          <select name="equipment">
            {% for g in ['No equipment (home)','Dumbbells only','Full gym'] %}<option {% if f.equipment == g %}selected{% endif %}>{{ g }}</option>{% endfor %}
          </select></div>
        <div style="grid-column: span 2;"><label>Diet preference</label>
          <select name="diet">
            {% for g in ['Vegetarian','Non-Vegetarian','Vegan'] %}<option {% if f.diet == g %}selected{% endif %}>{{ g }}</option>{% endfor %}
          </select></div>
      </div>
      <button type="submit">Generate My Plan</button>
    </form>
    {% if result %}
    <div class="result">
      <div class="engine">Generated by: {{ engine }}</div>
      {{ result | safe }}
    </div>
    {% endif %}
  </div>
</body>
</html>"""

with open("templates/index.html", "w") as f:
    f.write(html)
print("index.html created")

index.html created


**Step 3: app.py**

In this step, we create the app.py file to develop the backend of the FitBuddy application using FastAPI. It handles user requests, processes fitness details, generates fitness plans, and connects the backend with the HTML frontend.

In [24]:
code = '''
from fastapi import FastAPI, Request, Form
from fastapi.responses import HTMLResponse
from fastapi.templating import Jinja2Templates
from pathlib import Path
import os, markdown

app = FastAPI(title="Fit Buddy")
templates = Jinja2Templates(directory=str(Path.cwd() / "templates"))

GEMINI_MODEL = "gemini-2.5-flash"   # Model not found error vandha AI Studio-la latest flash model name paathu maathunga
_local = None


def render(request, result="", form=None, engine=""):
    ctx = {"result": result, "f": form or {}, "engine": engine}
    try:
        return templates.TemplateResponse(request, "index.html", ctx)
    except TypeError:
        ctx["request"] = request
        return templates.TemplateResponse("index.html", ctx)


def ask_ai(prompt):
    global _local
    key = os.environ.get("GEMINI_API_KEY", "").strip()
    if key:
        from google import genai
        client = genai.Client(api_key=key)
        r = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
        return r.text, "Gemini (" + GEMINI_MODEL + ")"
    if _local is None:
        from transformers import pipeline
        _local = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")
    out = _local([{"role": "user", "content": prompt}], max_new_tokens=500)
    return out[0]["generated_text"][-1]["content"], "Qwen local model (no key)"


@app.get("/", response_class=HTMLResponse)
async def home(request: Request):
    return render(request)


@app.post("/generate", response_class=HTMLResponse)
async def generate(
    request: Request,
    age: str = Form(""), gender: str = Form(""), weight: str = Form(""),
    height: str = Form(""), goal: str = Form(""), level: str = Form(""),
    days: str = Form("4"), equipment: str = Form(""), diet: str = Form(""),
):
    form = dict(age=age, gender=gender, weight=weight, height=height, goal=goal,
                level=level, days=days, equipment=equipment, diet=diet)
    prompt = f"""You are Fit Buddy, a friendly certified fitness coach.
Create a personalised plan for this person:
Age: {age}, Gender: {gender}, Weight: {weight} kg, Height: {height} cm
Goal: {goal}, Fitness level: {level}, Workout days per week: {days}
Equipment: {equipment}, Diet preference: {diet}

Give the answer in markdown with these sections:
1. BMI and a one-line summary
2. Weekly workout plan (day-wise, exercises with sets x reps, include rest days)
3. Simple one-day diet plan (breakfast, lunch, snack, dinner) with approximate calories
4. Five short tips
End with a one-line note to consult a doctor before starting."""
    try:
        text, engine = ask_ai(prompt)
        html = markdown.markdown(text, extensions=["tables"])
        return render(request, html, form, engine)
    except Exception as e:
        return render(request, "<b>Error:</b> " + str(e), form, "-")
'''

with open("app.py", "w") as f:
    f.write(code)
print("app.py created")

app.py created


**Step 4: Server start**

In this step, we start the FitBuddy application server using Uvicorn. The server runs the FastAPI backend and allows users to access the application through a web browser.

In [25]:
import sys, socket, threading, time, uvicorn

sys.modules.pop("app", None)
from app import app

s = socket.socket()
s.bind(("", 0))
PORT = s.getsockname()[1]
s.close()

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(5)
print("Server started on port", PORT)

Server started on port 37577


**Step 5: UI**

In this step, we design the user interface of the FitBuddy application using HTML and CSS. The interface allows users to enter their personal fitness details, select their fitness goals, and generate a personalized fitness plan. It provides a simple and user-friendly experience.

In [26]:
from google.colab import output
output.serve_kernel_port_as_iframe(PORT, height=800)

<IPython.core.display.Javascript object>